

---


# **DOMAIN ADAPTIVE AGENTIC LEGAL AI (COMPLIANCE SYSTEM)**


---


```
                            ARCHITECTURE OF THE PROJECT
                         -----------------------------------

                                 [ USER QUERY ]
                                         │
                     "What is the penalty for a data breach?"
                                         │
                                         ▼
                             ┌──────────────────────────┐
                             │ AGENT 1: SEMANTIC ROUTER │
                             │   (Intent Classification │
                             │   via Pydantic Schema)   │
                             └────┬─────────┬────────┬──┘
                                  │         │        │
             ┌────────────────────┘         │        └────────────────────┐
             ▼                              ▼                             ▼
    (Intent: Out-of-Domain)        (Intent: DPDP Act)             (Intent: IT Act)
             │                              │                             │
             │                              ▼                             ▼
             │                 ┌────────────────────────┐    ┌────────────────────────┐
             │                 │   DPDP HYBRID STORE    │    │    IT HYBRID STORE     │
             │                 │  - FAISS (Dense)       │    │  - FAISS (Dense)       │
             │                 │  - BM25 (Sparse)       │    │  - BM25 (Sparse)       │
             │                 └────────────┬───────────┘    └────────────┬───────────┘
             │                              │                             │
             │                       (Top 30 Chunks)               (Top 30 Chunks)
             │                              │                             │
             │                              └──────────────┬──────────────┘
             │                                             │
             │                                             ▼
             │                              ┌──────────────────────────────┐
             │                              │   RECIPROCAL RANK FUSION     │
             │                              │    & CROSS-ENCODER RERANKER  │
             │                              │  Refines 30 Chunks -> Top 5  │
             │                              └──────────────┬───────────────┘
             │                                             │
             │                                             ▼
             │                              ┌──────────────────────────┐
             │                              │ AGENT 2: DOCUMENT GRADER │
             │                              │  (Evaluates Top 5 Chunks │
             │                              │   and drops irrelevant)  │
             │                              └──────────────┬───────────┘
             │                                             │
             │                                     ┌───────┴───────┐
             │                                     ▼               ▼
             │                           [If ALL 5 Irrelevant]  [If ≥1 Relevant]
             │                                     │               │
             │                                     │               ▼
             │                                     │    ┌──────────────────────────┐ <──────┐
             │                                     │    │ AGENT 3: GENERATOR AGENT │        │
             │                                     │    │   (Qwen 2.5 on T4 GPU)   │        │
             │                                     │    └────────────┬─────────────┘        │
             │                                     │                 │                      │
             │                                     │                 ▼                      │
             │                                     │    ┌──────────────────────────┐        │
             │                                     │    │  AGENT 4: AUDITOR AGENT  │        │
             │                                     │    │ (Checks if the answer is │        │
             │                                     │    │ grounded in source docs, │        │
             │                                     │    │  matches the user query, │        │
             │                                     │    │  and is not hallucinated)│        │
             │                                     │    │     (Max Retries: 2)     │        │
             │                                     │    └────────────┬─────────────┘        │
             │                                     │                 │                      │
             │                                     │          ┌──────┴──────┐               │
             │                                     │          ▼             ▼               │
             │                                     │    [Passes Audit] [Fails Audit]        │
             │                                     │          │             │               │
             │                                     │          │             └───────────────┘
             │                                     │          │         (Triggers Re-generation)
             ▼                                     ▼          ▼
    ┌──────────────────────────┐                   │    ┌──────────────────────────────────┐
    │   SMART FALLBACK AGENT   │<──────────────────┘    │      GENERATED LEGAL ANSWER      │
    │ (Gracefully deflects     │                        │----------------------------------│
    │  query & suggests        │                        │ "According to the DPDP Act..."   │
    │  external sources)       │                        └──────────────────────────────────┘
    └────────────┬─────────────┘
                 │
                 ▼
    ┌──────────────────────────────────┐
    │      FINAL FALLBACK MESSAGE      │
    │----------------------------------│
    │ "I am specialized strictly in... │
    │ so this is out of my domain."    │
    └──────────────────────────────────┘
```



---


#### **AUTHENTICATION**

---

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# setting the path
path = '/content/drive/MyDrive/Colab_Notebooks/LegalAI_Updated'

# changing the directory and finding the pdf
try:
  os.chdir(path)
  print(f"Succesfully connected to: {os.getcwd()}")

  # getting the list of all files from the folder
  allFiles = os.listdir(path)

  # filtering the pdf files only from allFiles
  pdfFiles = [f for f in allFiles if f.lower().endswith('.pdf')]

  # if pdfs founded then print their names
  if pdfFiles:
    print("\nLegal documents names are following:-\n")
    for pdf in pdfFiles:
      print(f"{pdf}")
  else:
    print("There are no pdf available in LegalAI_Project_Data folder")

except FileNotFoundError:
  print("Path not found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Succesfully connected to: /content/drive/MyDrive/Colab_Notebooks/LegalAI_Updated

Legal documents names are following:-

DPDP_Act.pdf
IT_Act.pdf


---
#### **INSTALLATIONS**

---

In [ ]:
# 1. Core LangChain Packages
!pip install langchain langchain-community
!pip install -q langchain-classic
!pip install -q langchain
!pip install -q langchain-text-splitters

# 2. Document Parsing & Processing
!pip install -q unstructured "unstructured[pdf]"
!apt-get install -y poppler-utils
!pip install -q tiktoken

# 3. Vector Store & Sparse Search (Hybrid Setup)
!pip install -q faiss-cpu
!pip install -q rank_bm25

# 4. Embeddings & Deep Learning Models
!pip install -q langchain-huggingface
!pip install -q sentence-transformers
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes

# 5. Agentic Workflow
!pip install -q langgraph
!pip install -q semantic-router


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


---

---




# **IMPLEMENTATION PROCEDURE**


---

---

#### STEP 1

---


# **DATA INGESTION AND PREPROCESSING**

---

In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import tiktoken

dpdpFile = "DPDP_Act.pdf"
itFile = "IT_Act.pdf"

# Setting the files path
filesPath = '/content/drive/MyDrive/Colab_Notebooks/LegalAI_Updated'
os.chdir(filesPath)

# Function to ONLY load and clean the PDF
def load_and_clean_pdf(fileName):
    pdfPath = os.path.join(filesPath, fileName)

    # Check if pdf exists
    if not os.path.exists(pdfPath):
        print(f"File {fileName} does not exist")
        return []

    print(f"{fileName} is loading...")

    # Cleaning the pdf
    loader = UnstructuredPDFLoader(pdfPath, mode="single", strategy="fast")
    document = loader.load()

    for doc in document:
        # Joining broken words and preserving paragraphs
        doc.page_content = doc.page_content.replace("-\n\n", "")
        doc.page_content = doc.page_content.replace("-\n", "")
        doc.page_content = doc.page_content.replace("\n\n", "<<PARA>>")
        doc.page_content = doc.page_content.replace("\n", " ")
        doc.page_content = doc.page_content.replace("<<PARA>>", "\n\n")

    print(f"Cleaned {fileName} successfully.")
    return document

# execution
print("Data ingestion started...\n")

# Load full cleaned documents (Not chunks yet)
dpdp_docs = load_and_clean_pdf(dpdpFile)
it_docs = load_and_clean_pdf(itFile)

print("\nConfiguring Parent-Child Splitters...\n")

# 1. Parent Splitter: Large context for the LLM
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1024,
    chunk_overlap=100,
    # Priority for splitting to preserve text context
    separators=["\n\n", "\n", ".", " ", ""]
)

# 2. Child Splitter: Small context for highly accurate Vector Search
child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=256,
    chunk_overlap=25,
    separators=["\n\n", "\n", ".", " ", ""]
)

print("Step 1 Completed: Data Ingested and Splitters Configured.")

Data ingestion started...

DPDP_Act.pdf is loading...
Cleaned DPDP_Act.pdf successfully.
IT_Act.pdf is loading...
Cleaned IT_Act.pdf successfully.

Configuring Parent-Child Splitters...

Step 1 Completed: Data Ingested and Splitters Configured.


#### STEP 2

---
# **EMBEDDINGS AND VECTOR STORES**

---

In [ ]:
import os
import pickle
import faiss
import torch

from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

from langchain_community.retrievers import BM25Retriever


# Define Paths
filesPath = '/content/drive/MyDrive/Colab_Notebooks/LegalAI_Updated'
if not os.path.exists(filesPath):
    os.makedirs(filesPath)

# Check device availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}\n")

# loading the embedding model
print("Loading Advanced Embedding Model (BAAI/bge-large-en-v1.5)...")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)
print("Embedding Model loaded successfully.\n")

# hybrid store
def get_or_build_hybrid_store(docs, store_name, parent_splitter, child_splitter):

    # Define save paths for the 3 components
    faiss_path = os.path.join(filesPath, f"faiss_dense_{store_name}")
    docstore_path = os.path.join(filesPath, f"docstore_{store_name}.pkl")
    bm25_path = os.path.join(filesPath, f"bm25_sparse_{store_name}.pkl")

    # Check if all components already exist on Google Drive
    if os.path.exists(faiss_path) and os.path.exists(docstore_path) and os.path.exists(bm25_path):
        print(f"Found existing Hybrid Store for {store_name}. Loading into memory...")

        # Load FAISS
        vectorstore = FAISS.load_local(
            faiss_path,
            embeddings,
            allow_dangerous_deserialization=True
        )

        # Load DocStore
        with open(docstore_path, 'rb') as f:
            docstore = pickle.load(f)

        # Load BM25
        with open(bm25_path, 'rb') as f:
            bm25_retriever = pickle.load(f)

        # Reconstruct the ParentDocumentRetriever
        parent_retriever = ParentDocumentRetriever(
            vectorstore=vectorstore,
            docstore=docstore,
            child_splitter=child_splitter,
            parent_splitter=parent_splitter
        )
        return parent_retriever, bm25_retriever

    else:
        print(f"No existing store found for {store_name}. Building Gold Standard Hybrid Index...")
        if not docs:
            print("Error: No documents provided to create new store.")
            return None, None

        # Initialize an empty FAISS index
        # BGE-Large output dimension is 1024.
        # IndexFlatIP calculates the Inner Product (which equals Cosine Sim for normalized vectors)
        embedding_dim = 1024
        index = faiss.IndexFlatIP(embedding_dim)

        vectorstore = FAISS(
            embedding_function=embeddings,
            index=index,
            docstore=InMemoryDocstore(),
            index_to_docstore_id={}
        )

        # Initialize the Key-Value store for Parent Documents
        docstore = InMemoryStore()

        # Initialize the ParentDocumentRetriever framework
        parent_retriever = ParentDocumentRetriever(
            vectorstore=vectorstore,
            docstore=docstore,
            child_splitter=child_splitter,
            parent_splitter=parent_splitter
        )

        # Process documents: This splits docs into parents, splits parents into children,
        # embeds the children, and creates the UUID mapping under the hood.
        print(" -> Splitting documents and generating dense embeddings (This may take a minute)...")
        parent_retriever.add_documents(docs)

        # Retrieve the newly created Parent Documents to build the Sparse Index (BM25)
        print(" -> Building Sparse BM25 Keyword Index...")
        parent_keys = list(docstore.yield_keys())
        parent_docs = docstore.mget(parent_keys)

        # Create BM25 retriever directly from Parent Documents
        bm25_retriever = BM25Retriever.from_documents(parent_docs)

        # Save all three components to Google Drive for future runs
        print(" -> Saving indexes to disk...")
        vectorstore.save_local(faiss_path)
        with open(docstore_path, 'wb') as f:
            pickle.dump(docstore, f)
        with open(bm25_path, 'wb') as f:
            pickle.dump(bm25_retriever, f)

        print(f"Hybrid store created and saved successfully for {store_name}!\n")
        return parent_retriever, bm25_retriever

# pass the 'dpdp_docs' and splitters generated in Step 1
dpdp_parent_retriever, dpdp_bm25 = get_or_build_hybrid_store(
    docs=dpdp_docs,
    store_name="DPDP_Act",
    parent_splitter=parent_splitter,
    child_splitter=child_splitter
)

it_parent_retriever, it_bm25 = get_or_build_hybrid_store(
    docs=it_docs,
    store_name="IT_Act",
    parent_splitter=parent_splitter,
    child_splitter=child_splitter
)

print("Step 2 Completed: Dense (FAISS) and Sparse (BM25) Parent-Child Stores are ready.")

Using Device: cuda

Loading Advanced Embedding Model (BAAI/bge-large-en-v1.5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model loaded successfully.

Found existing Hybrid Store for DPDP_Act. Loading into memory...
Found existing Hybrid Store for IT_Act. Loading into memory...
Step 2 Completed: Dense (FAISS) and Sparse (BM25) Parent-Child Stores are ready.


#### STEP 3

---
# **IMPLEMENTING INTELLIGENT LLM SEMANTIC ROUTER**

---


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline

# loading the llm
print("Loading Qwen-2.5-7B-Instruct in 4-bit...")

modelId = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(modelId)
model = AutoModelForCausalLM.from_pretrained(modelId, quantization_config=bnb_config, device_map="auto")

# HuggingFace Pipeline
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=None,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.01,
    return_full_text=False,
    repetition_penalty=1.05,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

# Wrap pipeline in LangChain's LLM class
llm = HuggingFacePipeline(pipeline=hf_pipeline)
print("LLM Loaded Successfully!\n")

# defining pydantic schema
class RouteDecision(BaseModel):
    category: Literal["dpdp_act", "it_act", "none"] = Field(
        description="The classification category for the user's query."
    )
    reasoning: str = Field(
        description="A short 1-sentence logic why this category was chosen."
    )

# LangChain Parser setup
parser = PydanticOutputParser(pydantic_object=RouteDecision)

# structured prompt
router_prompt = PromptTemplate(
   template="""<|im_start|>system
You are an expert Legal Routing AI.
Analyze the user's query and classify it into one of the following categories:
- 'dpdp_act': For data privacy, data protection, personal data breach, DPDP Act 2023.
- 'it_act': For cyber crime, hacking, electronic signatures, IT Act 2000.
- 'none': For general chat, greetings, or out-of-domain questions.

{format_instructions}

CRITICAL RULES:
- Output ONLY a valid JSON object.
- DO NOT use markdown like ```json.
- DO NOT add conversational text.
- LANGUAGE LOCK: Ensure the 'reasoning' value is strictly in English. Do NOT output any Chinese characters.<|im_end|>
<|im_start|>user
User Query: {query}<|im_end|>
<|im_start|>assistant
""",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create the LangChain Execution Chain
router_chain = router_prompt | llm | parser

# llm based router function (AGENT 1)
def structured_llm_router(query):
    try:
        result = router_chain.invoke({"query": query})
        print(f"  [Reasoning Log]: {result.reasoning}")
        return result.category

    except Exception as e:
        print(f"  [Parsing Error - Defaulting to 'none']: {e}")
        return "none"

# testing the router
print("Testing Router...\n")

test_queries = [
    "What is the penalty for a data breach?",
    "Is hacking a bailable offence?",
    "Tell me a python code for a calculator"
]

for q in test_queries:
    print(f"Query: '{q}'")
    decision = structured_llm_router(q)
    print(f"➔ Route Decision: {decision}\n")

print("Step 3 Completed: Structured Router is Active!")

Loading Qwen-2.5-7B-Instruct in 4-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample', 'max_length', 'eos_token_id', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM Loaded Successfully!

Testing Router...

Query: 'What is the penalty for a data breach?'
  [Reasoning Log]: The question pertains to the penalties associated with data breaches, which falls under data privacy laws.
➔ Route Decision: dpdp_act

Query: 'Is hacking a bailable offence?'
  [Reasoning Log]: The question pertains to cyber crime, which falls under the IT Act 2000.
➔ Route Decision: it_act

Query: 'Tell me a python code for a calculator'
  [Reasoning Log]: The user's query is about a Python code for a calculator, which is not related to data privacy, data protection, personal data breach, IT Act 2000, cyber crime, or hacking.
➔ Route Decision: none

Step 3 Completed: Structured Router is Active!


#### STEP 4

---
# **HYBRID RETRIEVAL OF DOCUMENTS**

---

In [ ]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever

# cross encoder model
print("Loading Cross-Encoder Model (bge-reranker-v2-m3)...")

cross_encoder_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",
    model_kwargs={"max_length": 1024}
)

reranker_compressor = CrossEncoderReranker(model=cross_encoder_model, top_n=5)
print("Reranker Loaded Successfully!\n")

# hybrid reranking pipeline
def advanced_hybrid_rerank_pipeline(query, route_decision):

    if route_decision == "none":
        return []

    print(f"  -> Executing RRF + Reranking Pipeline in: {route_decision.upper()}")

    # Select Retrievers
    if route_decision == "dpdp_act":
        dense_retriever = dpdp_parent_retriever
        sparse_retriever = dpdp_bm25
    elif route_decision == "it_act":
        dense_retriever = it_parent_retriever
        sparse_retriever = it_bm25
    else:
        return []

    # Setup both retrievers to fetch a BROADER pool of chunks (Top 30 each)
    dense_retriever.search_kwargs = {"k": 30}
    sparse_retriever.k = 30

    # RRF (EnsembleRetriever)
    hybrid_rrf_retriever = EnsembleRetriever(
        retrievers=[dense_retriever, sparse_retriever],
        weights=[0.5, 0.5]
    )

    # Long-Context Reranker
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=reranker_compressor,
        base_retriever=hybrid_rrf_retriever
    )

    print("  -> Running BM25 + FAISS + RRF + Cross-Encoder...")
    final_top_5_docs = compression_retriever.invoke(query)

    return final_top_5_docs

Loading Cross-Encoder Model (bge-reranker-v2-m3)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker Loaded Successfully!



#### STEP 5

---
# **AGENTIC RAG (SELF-CORRECTION)**

---
a) DEFINING THE AGENTS

In [ ]:
from typing import List, TypedDict
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langgraph.graph import END, StateGraph

# graph state definition
class AgenticRAGState(TypedDict):
    query: str
    route: str
    documents: List[str]
    generation: str
    retries: int

# AGENT 2: DOCUMENT GRADER
class GradeDocuments(BaseModel):
    binary_score: str = Field(description="Are the documents relevant to the question? Answer 'yes' or 'no'")
    reasoning: str = Field(description="Short reasoning for your decision")

doc_grader_parser = PydanticOutputParser(pydantic_object=GradeDocuments)

doc_grader_prompt = PromptTemplate(
   template="""<|im_start|>system
You are an expert legal auditor.
TASK: Check if the document is relevant to answering the User Question ('yes' or 'no').

{format_instructions}

CRITICAL RULES:
- Output ONLY a valid JSON object.
- DO NOT use markdown like ```json.
- DO NOT continue any lists or write numbers.
- DO NOT add explanations outside the JSON.
- LANGUAGE LOCK: Ensure the 'reasoning' value is strictly in English. Do NOT output any Chinese characters.

EXAMPLE OUTPUT FORMAT:
{{"binary_score": "yes", "reasoning": "Contains information about penalties."}}<|im_end|>
<|im_start|>user
--- START DOCUMENT ---
{document}
--- END DOCUMENT ---

User Question: "{query}"

Now, generate the JSON for the document above:<|im_end|>
<|im_start|>assistant
""",
    input_variables=["query", "document"],
    partial_variables={"format_instructions": doc_grader_parser.get_format_instructions()},
)
doc_grader_chain = doc_grader_prompt | llm | doc_grader_parser

# AGENT 3: THE LEGAL GENERATOR
gen_prompt = PromptTemplate(
    template="""<|im_start|>system
You are an expert yet accessible Indian Legal Advisor.
Carefully analyze the provided legal context and answer the user's question.

CRITICAL INSTRUCTIONS:
- ACCURACY FIRST: Base your answer STRICTLY on the provided Context. Do not include any outside knowledge.
- If the context contains multiple related clauses, conditions, or numbers, cross-examine them carefully before formulating your conclusion.
- PLAIN ENGLISH: Write your answer in simple, everyday English so that a normal person with no legal background can easily understand it. Translate complex legal jargon into simple terms.
- CONCISE & STRUCTURED: Keep the response to a standard, readable length. Summarize the points clearly using bullet points for better readability. Do not output a massive wall of text.
- If the exact answer is not available in the context, explicitly state "I do not have enough information from the provided legal texts to answer this."
- Provide a clear, clean and readable response.
- LANGUAGE LOCK: You MUST write the entire response strictly in English. Do NOT output any Chinese characters.<|im_end|>
<|im_start|>user
Context:
--- START CONTEXT ---
{context}
--- END CONTEXT ---

Question: {query}<|im_end|>
<|im_start|>assistant
""",
    input_variables=["context", "query"]
)
generator_chain = gen_prompt | llm | StrOutputParser()

# AGENT 4: HALLUCINATION & RELEVANCE AUDITOR
class GradeGeneration(BaseModel):
    is_grounded: str = Field(description="Is the answer strictly based on the facts provided? Answer 'yes' or 'no'")
    is_relevant: str = Field(description="Does the answer directly address the user's question? Answer 'yes' or 'no'")
    reasoning: str = Field(description="Short reasoning")

gen_grader_parser = PydanticOutputParser(pydantic_object=GradeGeneration)

gen_grader_prompt = PromptTemplate(
 template="""<|im_start|>system
You are a strict Legal Auditor evaluating an AI's answer.
TASK: Evaluate the AI ANSWER based strictly on the FACTS provided.
- is_grounded: Is the underlying meaning completely based on the FACTS without adding outside info? ('yes' or 'no').
  IMPORTANT RULE: The AI is instructed to write in simple, plain English and use bullet points for users. DO NOT fail or penalize the AI for using simple vocabulary, paraphrasing, or formatting. Evaluate ONLY if the core FACTS are correct.
- is_relevant: Does it directly answer the User Question accurately? ('yes' or 'no')

{format_instructions}

CRITICAL RULES:
- Output ONLY a valid JSON object.
- DO NOT use markdown like ```json.
- DO NOT add explanations outside the JSON.
- STRICT FACT & NUMBER CHECK: If the AI ANSWER contains any numbers, penalty amounts (e.g., crores), or section numbers, they MUST perfectly match the FACTS.
- Focus strictly on factual accuracy, NOT on the simplicity of the language or the use of bullet points.
- LANGUAGE LOCK: Ensure the 'reasoning' value is strictly in English. Do NOT output any Chinese characters.

EXAMPLE OUTPUT FORMAT:
{{"is_grounded": "yes", "is_relevant": "yes", "reasoning": "state clearly why it passes or fails. E.g., 'The penalty amount matches the facts' OR 'The AI hallucinated a penalty of 500 crore which contradicts the facts."}}<|im_end|>
<|im_start|>user
FACTS:
--- START FACTS ---
{documents}
--- END FACTS ---

User Question: "{query}"

AI ANSWER:
--- START ANSWER ---
{generation}
--- END ANSWER ---

JSON OUTPUT:<|im_end|>
<|im_start|>assistant
""",
    input_variables=["documents", "generation", "query"],
    partial_variables={"format_instructions": gen_grader_parser.get_format_instructions()},
)
gen_grader_chain = gen_grader_prompt | llm | gen_grader_parser

print("All Agent Chains Initialized!")

All Agent Chains Initialized!


b) LANGRAPH WORKFLOW ASSEMBLY

In [ ]:
import torch
import gc

# node functions
def retrieve_docs(state: AgenticRAGState):
    print("\n[TOOL] Retriever: Fetching information from Hybrid Database...")
    query = state["query"]
    route = state["route"]

    docs = advanced_hybrid_rerank_pipeline(query, route)
    doc_texts = [d.page_content for d in docs]
    return {"documents": doc_texts, "retries": state.get("retries", 0)}

def grade_docs_node(state: AgenticRAGState):
    print("\n[AGENT 2] Document Grader: Filtering irrelevant chunks...")
    query = state["query"]
    documents = state["documents"]

    filtered_docs = []

    for i, d in enumerate(documents):
        print(f"\n   Evaluating Document {i+1}/{len(documents)}...")
        try:
            result = doc_grader_chain.invoke({"query": query, "document": d})

            # if doc is relevant
            if "yes" in result.binary_score.lower():
                print(f"   -> [Verdict: RELEVANT] Keeping chunk. Reason: {result.reasoning}")
                filtered_docs.append(d)
            # if doc irrelevant (filtered out)
            else:
                print(f"   -> [Verdict: IRRELEVANT] Filtering out chunk. Reason: {result.reasoning}")

        except Exception as e:
            # fallback if zephyr json failed while parsing
            print(f"   -> [Verdict: ERROR] Parsing failed, keeping as fallback. Error details:", {e})
            filtered_docs.append(d)

    print(f"\n       -> Final Result: Kept {len(filtered_docs)} out of {len(documents)} chunks.")
    return {"documents": filtered_docs}


def generate_node(state: AgenticRAGState):
    print("\n[AGENT 3] Generator: Drafting the final legal response...")

    # Clear VRAM before LLM starts thinking
    gc.collect()
    torch.cuda.empty_cache()

    query = state["query"]
    documents = state["documents"]

    current_retries = state.get("retries", 0)

    # Restrict to Top 3 chunks to prevent Attention Matrix explosion
    safe_documents = documents[:3]
    print(f"       -> Using top {len(safe_documents)} chunks to save GPU Memory.")

    context_str = "\n\n".join(safe_documents)

    # Generate the answer
    generation = generator_chain.invoke({"context": context_str, "query": query})

    # Clear VRAM again after generation
    torch.cuda.empty_cache()

    return {
        "generation": generation,
        "retries": current_retries + 1
    }

# edge routing function
def decide_to_generate(state: AgenticRAGState):
    print("\n[EDGE] Checking if we have valid context to proceed...")
    if not state["documents"]:
        print("       -> No relevant documents found. Stopping.")
        return "end"
    return "generate"

def grade_generation_edge(state: AgenticRAGState):
    print("\n[AGENT 4] Auditor: Auditing the drafted response for hallucinations...")
    query = state["query"]
    documents = state["documents"]
    generation = state["generation"]
    retries = state.get("retries", 0)

    if retries >= 2:
        print("       -> Max retries reached. Outputting best effort.")
        return "useful"

    try:
        # giving Auditor only the top 3 chunks that the Generator used
        safe_documents = documents[:3]
        context_str = "\n\n".join(safe_documents)

        result = gen_grader_chain.invoke({
            "documents": context_str,
            "generation": generation,
            "query": query
        })

        print(f"       -> Grounded: {result.is_grounded.upper()} | Relevant: {result.is_relevant.upper()}")
        print(f"       -> Reasoning: {result.reasoning}")

        if "yes" in result.is_grounded.lower() and "yes" in result.is_relevant.lower():
            return "useful"
        else:
            print("       -> Answer failed audit! Forcing regeneration...")
            return "not_useful"

    except Exception as e:
        print(f"       -> Audit parsing error! Exact Details: {e}")
        print("       -> Passing by default to prevent system crash.")
        return "useful"

# compiling the graph
workflow = StateGraph(AgenticRAGState)

workflow.add_node("retrieve", retrieve_docs)
workflow.add_node("grade_documents", grade_docs_node)
workflow.add_node("generate", generate_node)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")

# Edge 1: if docs are correct then generate otherwise stop
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "generate": "generate",
        "end": END,
    }
)

# Edge 2: Agentic loop - if hallucinated answer then return to generate step
workflow.add_conditional_edges(
    "generate",
    grade_generation_edge,
    {
        "useful": END,
        "not_useful": "generate",
    }
)

legal_ai_app = workflow.compile()
print("Agentic RAG Graph Compiled Successfully!")

Agentic RAG Graph Compiled Successfully!


#### STEP 6

---
# **USER FRIENDLY UI**

---

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import textwrap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# smart fallback prompt
fallback_prompt = PromptTemplate(
    template="""<|im_start|>system
You are a Domain Adaptive Legal AI specialized ONLY in the DPDP Act 2023 and IT Act 2000 of India.
The user asked a query that is OUTSIDE your domain.

CRITICAL INSTRUCTIONS:
- You must reply with EXACTLY TWO sentences.
- SENTENCE 1 MUST BE EXACTLY: "I am a Domain Adaptive Legal AI specialized strictly in the DPDP Act 2023 and IT Act 2000, so this query is out of my domain."
- SENTENCE 2 MUST BE a brief 1-line suggestion on where the user might find the answer (e.g., "For Python code, please check the official Python documentation or use a general AI like ChatGPT.").
- DO NOT add any greetings, extra explanations, or repetitions. Stop generating immediately after Sentence 2.
- LANGUAGE LOCK: You MUST generate the entire response strictly in English. Do NOT output any Chinese characters.<|im_end|>
<|im_start|>user
User Query: {query}<|im_end|>
<|im_start|>assistant
""",
    input_variables=["query"]
)

fallback_chain = fallback_prompt | llm | StrOutputParser()

# sample questions
query_counter = 0

sample_questions = [
    "What is the maximum penalty for a significant data breach?",
    "What are the duties of a Data Principal under the 2023 Act?",
    "What is the penalty for hacking a computer system?",
    "Are electronic signatures legally recognized in court?",
    "Is there a 250 crore penalty for hacking a system under the IT Act?",
    "If a hospital processes data for a medical emergency, do they need consent?",
    "Can a child under 18 provide their own consent for data processing?",
    "What happens if someone publishes obscene material in electronic form?",
    "Tell me a python code to build a calculator.",
    "What is the best recipe for an authentic Indian chicken or mutton curry?",
    "What are the traffic rules for jumping a red light?"
]

# ui components
title_html = widgets.HTML("<h2>🏛️<b>DOMAIN ADAPTIVE AGENTIC LEGAL AI</b></h2><p style='color:gray;'>Powered by Qwen 2.5 & LangGraph</p><hr>")

query_input = widgets.Textarea(
    value='',
    placeholder='Type your custom legal query here...',
    description='Query:',
    layout={'width': '80%', 'height': '60px'}
)
submit_btn = widgets.Button(description="Submit Query", button_style='primary')
clear_btn = widgets.Button(description="Clear Output", button_style='warning')

predefined_dropdown = widgets.Dropdown(
    options=['--- Select a Sample Question ---'] + sample_questions,
    value='--- Select a Sample Question ---',
    description='Samples:',
    layout={'width': '80%'}
)

response_box_title = widgets.HTML("<h3 style='margin-top: 30px; margin-bottom: 5px; color: white; font-family: sans-serif;'><b>AI RESPONSE</b></h3>")

output_area = widgets.Output(layout={
    'border': '1px solid gray',
    'padding': '15px',
    'height': '550px',
    'overflow_y': 'auto'
})

# procesing function
def process_query(query):
    global query_counter

    if not query.strip():
        return

    query_counter += 1

    with output_area:
        display(HTML(f"<div style='background-color: #ecfdf5; border-left: 6px solid #10b981; padding: 12px; border-radius: 6px; font-size: 16px; color: #064e3b; margin-top: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);'><b>👤 USER QUERY NO {query_counter}:</b> {query}</div>"))

        print("\n" + "="*70)
        print(f"[AGENT 1] Semantic Router: Classifying Query No {query_counter}...")

        try:
            route = structured_llm_router(query)
            print(f"  -> Query routed To: {route.upper()}")

            final_generation = ""
            if route != "none":
                inputs = {"query": query, "route": route, "retries": 0}

                for output in legal_ai_app.stream(inputs):
                    for key, value in output.items():
                        if "generation" in value:
                            final_generation = value["generation"]

               if not final_generation.strip():
                    print("  -> All retrieved documents were irrelevant. Generating Smart Fallback...")
                    final_generation = fallback_chain.invoke({"query": query})
            else:
                print("  -> Query classified as Out-of-Domain. Generating Smart Fallback...")
                final_generation = fallback_chain.invoke({"query": query})

        except Exception as e:
             final_generation = f"An error occurred: {e}"
             print(f"  -> Error: {e}")

        print("="*70 + "\n")

        formatted_answer = textwrap.fill(final_generation, width=95).replace('\n', '<br>')

        display(HTML(f"<div style='background-color: #ecfdf5; border-left: 6px solid #10b981; padding: 18px; border-radius: 6px; font-size: 16px; color: #064e3b; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); line-height: 1.5;'><b>🏛️ FINAL LEGAL ANSWER FOR QUERY NO {query_counter}:</b><br><br>{formatted_answer}</div>"))

# event handlers
def on_submit_clicked(b):
    query = query_input.value.strip()
    if not query and predefined_dropdown.value != '--- Select a Sample Question ---':
        query = predefined_dropdown.value

    if query:
        query_input.value = ''
        predefined_dropdown.value = '--- Select a Sample Question ---'
        process_query(query)

def on_dropdown_change(change):
    if change['new'] != '--- Select a Sample Question ---':
        query_input.value = change['new']

def on_clear_clicked(b):
    global query_counter
    query_counter = 0
    with output_area:
        clear_output()

submit_btn.on_click(on_submit_clicked)
clear_btn.on_click(on_clear_clicked)
predefined_dropdown.observe(on_dropdown_change, names='value')

# displaying ui sequences
input_row = widgets.HBox([query_input, submit_btn, clear_btn])

display(title_html, predefined_dropdown, input_row, response_box_title, output_area)

HTML(value="<h2>🏛️<b>DOMAIN ADAPTIVE AGENTIC LEGAL AI</b></h2><p style='color:gray;'>Powered by Qwen 2.5 & Lan…

Dropdown(description='Samples:', layout=Layout(width='80%'), options=('--- Select a Sample Question ---', 'Wha…

HTML(value="<h3 style='margin-top: 30px; margin-bottom: 5px; color: white; font-family: sans-serif;'><b>AI RES…

Output(layout=Layout(border='1px solid gray', height='550px', overflow_y='auto', padding='15px'))